In [1]:
from weakref import ref

import gdstk
import numpy
import numpy as np
from types import SimpleNamespace
import pathlib
import sys
import os
#um units, 0.1nm precision
#lib = gdstk.Library(unit=1e-6, precision=1e-9)

# 3. Load the template into a temporary space
lib = gdstk.read_gds(r"C:\Users\Connor\Documents\GDSPhotonics\singlesidedtemplate.gds")
# 1. Create your Global Parent
# Check if it exists first to avoid the "Duplicate Name" crash from before
if "GLOBALPARENT" in lib.cells:
    global_parent = lib.cells["GLOBALPARENT"]
else:
    global_parent = lib.new_cell("GLOBALPARENT")
parentglobal = SimpleNamespace(
    name = "GLOBALPARENT",
    cell = global_parent
)
existing_tops = [c for c in lib.top_level() if c.name != "GLOBALPARENT"]
for sub_cell in existing_tops:
    # Get the raw polygons out of the template cells
    # We use get_polygons() to catch everything in the sub-hierarchy
    raw_polys = sub_cell.get_polygons(include_paths=True)
    
    # Add them directly to global_parent as real, editable polygons
    global_parent.add(*raw_polys)

#existing_tops = [c for c in lib.top_level() if c.name != "GLOBALPARENT"]
#for cell in existing_tops:
    
    #ref = gdstk.Reference(cell, origin=(0, 0))
    #global_parent.add(ref)

# Define Devices 


In [2]:
#photonic crystal device parameters are defined here
device1 = SimpleNamespace(
    name="DEV1",
    beam_length = 45,
    beam_width = 1.7,
    beam_layer=1,
    ax=1.1323,
    d_max=0.117565,
    cavity_hole_num=12,
    d_max_rx=0,
    d_max_ry=0,
    rx=0.35354,
    ry=0.35354,
    holes_layer=1
)
device2 = SimpleNamespace(
    name="DEV2",
    beam_length = 45,
    beam_width = 1.7,
    beam_layer=1,
    ax=1.150792,
    d_max=0.1888,
    cavity_hole_num=12,
    d_max_rx=0,
    d_max_ry=0.4,
    rx=0.2574,
    ry=0.5205,
    holes_layer=1
)
device3 = SimpleNamespace(
    name="DEV3",
    beam_length = 45,
    beam_width = 1.7,
    beam_layer=1,
    ax=1.12426,
    d_max=0.20747,
    cavity_hole_num=12,
    d_max_rx=0,
    d_max_ry=0.5,
    rx=0.1895,
    ry=0.3936,
    holes_layer=1
)
device4 = SimpleNamespace(
    name="DEV4",
    beam_length = 45,
    beam_width = 1.7,
    beam_layer=1,
    ax=1.1845839,
    d_max=0.207477,
    cavity_hole_num=12,
    d_max_rx=0.3217,
    d_max_ry=0,
    rx=0.2979,
    ry=0.44861,
    holes_layer=1
)
device5 = SimpleNamespace(
    name="DEV5",
    beam_length = 45,
    beam_width = 1.7,
    beam_layer=1,
    ax=1.22944,
    d_max=0.207477,
    cavity_hole_num=12,
    d_max_rx=0.1648,
    d_max_ry=0.1648,
    rx=0.3428,
    ry=0.3428,
    holes_layer=1
)

#grating couplers defined here
gratingminus10 = SimpleNamespace(
filepath = r"C:\Users\Connor\Downloads\coupler_minus10.gds",
length = 4.68
)

grating = SimpleNamespace(
filepath = r"C:\Users\Connor\Downloads\coupler_og.gds",
length = 5.2
)

gratingplus10 = SimpleNamespace(
filepath = r"C:\Users\Connor\Downloads\coupler_plus10.gds",
length = 5.72
)

# Define Device Arrays


In [3]:
#from types import SimpleNamespace

# A-group: ARRAY1, ARRAY11, ..., ARRAY111111111  (no 'A' in the name)






for n in range(1, 11):
    suffix = "1" * n
    var_name = f"A{suffix}"         # A1, A11, A111, ...
    cell_name = f"ARRAY{suffix}"    # ARRAY1, ARRAY11, ...
    globals()[var_name] = SimpleNamespace(
        name=cell_name,
        cell=lib.new_cell(cell_name)
    )
    #parentglobal.cell.add(cell_name)
    ref = gdstk.Reference(globals()[var_name].cell, origin=(0, 0)) 
    import math
    #ref.rotation = math.pi / 2  # 90 degrees in radians
    parentglobal.cell.add(ref) 

# B–E groups: ARRAYB1, ARRAYB11, ..., ARRAYB111111111, etc.
for prefix in ["B", "C", "D", "E", "F", "J", "Q", "R"]:
    for n in range(1, 11):
        suffix = "1" * n
        var_name = f"{prefix}{suffix}"             # e.g. B1, B11, B111, ...
        cell_name = f"ARRAY{prefix}{suffix}"       # e.g. ARRAYB1, ARRAYB11, ...
        globals()[var_name] = SimpleNamespace(
            name=cell_name,
            cell=lib.new_cell(cell_name)
        )
        ref = gdstk.Reference(globals()[var_name].cell, origin=(0, 0)) 
        import math
        #ref.rotation = math.pi / 2  # 90 degrees in radians
        parentglobal.cell.add(ref) 
#existing_top.add(gdstk.Reference(A1.cell))


# Functions


In [4]:
"""
createNanoBeam()
Creates a bare nanobeam rectangle, centered at (0,0)
Micron units
Parameters:

length_um- length in um
width_um- width in um
layer- layer

Returns:

Single nanobeam rectangle

"""

def createNanobeam(length_um, width_um, layer=1):
    half_L = length_um / 2.0
    half_W = width_um / 2.0
    # Rectangle corners: (x0, y0) -> (x1, y1)
    x0, x1 = -half_L, half_L
    y0, y1 = -half_W, half_W
    beam_poly = gdstk.rectangle((x0, y0), (x1, y1), layer=layer)
    return beam_poly


In [5]:
"""
createHoles()
Creates elliptical cavity holes
Micron units 
Coordinates reference to cavity center at (0,0)
Parameters:

mirror_hole_num- number of mirror holes
device simpleNameSpace(kind of like a struct) containing:
ax-cavity hole spacing
d_max-cubic tapering parameter for spacing
cavity_hole_num- number cavity holes
dmaxry- ry cubic tapering parameter
dmaxrx- rx cubic tapering parameter
x is the longitudinal direction(parallel to beam)
y is transverse direction parallel to chip surface

Returns:
holes, an array of gdstk ellipses
"""

def createHoles(params, mirrorholes):
    ax = params.ax
    d_max = params.d_max
    cavity_hole_num = params.cavity_hole_num
    mirror_hole_num = mirrorholes
    d_max_rx = params.d_max_rx
    d_max_ry= params.d_max_ry
    rx= params.rx
    ry= params.ry
    layer = params.holes_layer
    holes = []

    #Calculate Lattice Spacing Tapering in Cavity Region
    if cavity_hole_num > 0:
        a_cav = [0.0] * cavity_hole_num
        for ii in range(1, cavity_hole_num + 1):
            t = ii / cavity_hole_num  # ii/cavity_hole_num
            A = 1 - d_max * abs(2 * (t**3) - 3 * (t**2) + 1)
            if ii == 1:
                # lumerical: a_cav(1) = ax*A/2, different here bc python different with matricies 
                a_cav[0] = ax * A / 2.0
            else:
                # lumerical:a_cav(ii) = ax*A, different here bc python different with matricies 
                a_cav[ii - 1] = ax * A
    else:
        a_cav = []

    #Radial Tapering in Cavity Region 
    if cavity_hole_num > 0:
        rx_cav = [0.0] * cavity_hole_num
        ry_cav = [0.0] * cavity_hole_num
        for ii in range(1, cavity_hole_num + 1):
            t = ii / cavity_hole_num
            Q = 1 - d_max_rx * abs(2 * (t**3) - 3 * (t**2) + 1)
            QQ = 1 - d_max_ry * abs(2 * (t**3) - 3 * (t**2) + 1)

            #lumerical: rx_cav(ii) = rx*Q;  ry_cav(ii) = ry*QQ;, again just different bc python has different matrix incidies
            rx_cav[ii - 1] = rx * Q
            ry_cav[ii - 1] = ry * QQ
    else:
        rx_cav = []
        ry_cav = []

    #Device Size Calculation
    cavity_size = 0.0
    if cavity_hole_num > 0:
        for ii in range(1, cavity_hole_num + 1):
            cavity_size = cavity_size + a_cav[ii - 1]

    mirror_size = mirror_hole_num * ax

    # device half-size
    device_size = cavity_size + mirror_size  # from cavity center to the last hole center

    #total hole number of half-device 
    device_hole_num = cavity_hole_num + mirror_hole_num

    #Hole Positioning 
    for ii in range(-device_hole_num, device_hole_num + 1):
        if ii == 0:
            # lumerical: if (ii == 0) {}
            continue

        abs_ii = abs(ii)
        sgn = 1.0 if ii > 0 else -1.0

        #Cavity Region
        if abs_ii <= cavity_hole_num and cavity_hole_num > 0:
            # calculate distance from cavity center
            dis_from_center = 0.0
            radialx = 0.0
            radialy = 0.0
            #again different from lumerical bc indexes 
            for jj in range(1, abs_ii + 1):
                dis_from_center = dis_from_center + a_cav[jj - 1]
                radialx = rx_cav[jj - 1]
                radialy = ry_cav[jj - 1]

            cx = sgn * dis_from_center
            cy = 0.0

            #add cavity hole to array 
            hole = gdstk.ellipse((cx, cy), (radialx, radialy), layer=layer, tolerance = 1e-5)
            holes.append(hole)

        #mirror region
        elif abs_ii <= (cavity_hole_num + mirror_hole_num):
            
            cx = sgn * (cavity_size + (abs_ii - cavity_hole_num) * ax)
            cy = 0.0
            #add mirror hole to array 
            hole = gdstk.ellipse((cx, cy), (rx, ry), layer=layer, tolerance=1e-5)
            holes.append(hole)

    return holes


In [6]:
"""
createWaveguideextension()
This function creates extensions to the waveguide for a device,
it starts the extension at the edge of the beam length, starts a taper from the beam
width to 1.7um if these are not the same value, ends the taper at taper_distance, and
ends the extension at waveguide_length
Units of microns

Parameters:

device: device namespace to have the extension added
waveguide length: length of extension
taper distance: distance the tapering is brought out to in a mismatch case 
double sided: choosing a 1 makes an extension on both sides, choosing a 0 makes an extension only on the left side, choosing -1 makes no extensions

Returns:

Polygons objects for extensions
"""


def createWaveguideextension(device, waveguide_length, taper_distance, double_sided):
   

    polys = []

    # No extensions
    if double_sided == -1 or waveguide_length <= 0 or taper_distance <= 0:
        return polys

    L = device.beam_length
    W_in = device.beam_width
    W_out = 1.7  # fixed
    layer = device.beam_layer

    #left side extension
    x0 = -L / 2.0                         
    x1 = x0 - taper_distance              
    x2 = x0 - waveguide_length            

    #Taper region
    left_taper = gdstk.Polygon(
        [
            (x0, +W_in/2.0),
            (x1, +W_out/2.0),
            (x1, -W_out/2.0),
            (x0, -W_in/2.0),
        ],
        layer=layer,
    )
    polys.append(left_taper)

    #Constant width
    left_const = gdstk.Polygon(
        [
            (x1, +W_out/2.0),
            (x2, +W_out/2.0),
            (x2, -W_out/2.0),
            (x1, -W_out/2.0),
        ],
        layer=layer,
    )
    polys.append(left_const)

    #right side extension
    if double_sided == 1:
        x0 =  L / 2.0
        x1 = x0 + taper_distance
        x2 = x0 + waveguide_length

        right_taper = gdstk.Polygon(
            [
                (x0, +W_in/2.0),
                (x1, +W_out/2.0),
                (x1, -W_out/2.0),
                (x0, -W_in/2.0),
            ],
            layer=layer,
        )
        polys.append(right_taper)

        right_const = gdstk.Polygon(
            [
                (x1, +W_out/2.0),
                (x2, +W_out/2.0),
                (x2, -W_out/2.0),
                (x1, -W_out/2.0),
            ],
            layer=layer,
        )
        polys.append(right_const)

    return polys


In [7]:
"""
createCoupler()
Loads a GDS path, assigns a layer, offset, and mirroring
Parameters:

filepath- path to coupler file 

layer-intended layer for coupler 
mirror- 1 to flip the device, 0 to keep it the same 

Returns:

coupler polygon
"""
def createCoupler(filepath, offset, mirror, target_layer, target_datatype):
   

    lib_tmp = gdstk.read_gds(filepath)

    coupler_poly = None

   
    for cell in lib_tmp.cells:
        for poly in cell.polygons:
            #if poly.layer == target_layer and poly.datatype == target_datatype:
            if poly.layer == target_layer:
                coupler_poly = poly.copy()
                break
        if coupler_poly is not None:
            break
   
    
   
    if mirror == 1:
        coupler_poly = coupler_poly.mirror((0,0), (0,1))
    
    coupler_poly = coupler_poly.translate(offset, 0)
    
   

    
    

    return coupler_poly


   



In [8]:
"""
placeDevice()
Places a single photonic crystal cavity device within a device array
Units of microns
(0,0) is center of device 
Parameters:
parent: device array, contains array name and parent cell to reference devices to
device: namespace for photonic device 
mirrorholes: # of mirror holes to create device with 
x- x position of device referenced to array
y- y position of device referenced to array
extensionparam- 1, 0 or -1, for double, left, or no sided extensions
extension_length - length of extensions
taperdistance- tapering distance of extensions
coupler-namespace for selected grating coupler 

returns:
Adds device with specified mirror holes, waveguide extensions/tapers, and grating coupler at position (x,y) to the parent cell

"""

def placeDevice(parent_cell, device, mirrorholes, x, y, extensionparam, extension_length, taperdistance, coupler):
    # Unique cell name of the form "ARRAYA_DEVB_MC_X_Y"
    cell_name = f"{parent_cell.name}__{device.name}__M{mirrorholes}_{x}_{y}"
    device_cell = lib.new_cell(cell_name)

    # nanobeam
    beam = createNanobeam(
        length_um=device.beam_length,
        width_um=device.beam_width,
        layer=1
    )

    
    holes = createHoles(device, mirrorholes)
 
    

    beamcut = gdstk.boolean([beam], holes, "not",1e-8, layer=3)

    
    # Waveguide extensions and couplers
    extensions = createWaveguideextension(device, extension_length, taperdistance, extensionparam)
    filepath = coupler.filepath
    couplength = coupler.length
    coupleroffset = 0.5*device.beam_length + extension_length + 0.5*couplength
   
        
    #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
    #coupler_polys1.layer = 6
    
        
    coupler_polys2 = createCoupler(filepath, -coupleroffset, 1, 4, 0)
    coupler_polys2.layer = 7

    # -------------------------------------------------
    # 1) Merge beam + extensions first (no coupler)
    # -------------------------------------------------
    core_wg = gdstk.boolean(
        beamcut + extensions,
        [],
        "or",
        1e-9
    )

    # -------------------------------------------------
    # 2) Build taper mask (modify as needed)
    # -------------------------------------------------
    taperdistance = 9
    taper_len   = taperdistance
    tip_width   = 0.2   # <-- your chosen final width (change this)

    # Rightmost edge of waveguide body (before coupler)
    x_end = 0.5 * device.beam_length + extension_length

    # Start of taper
    x_start = x_end - taper_len

    # Leftmost edge of waveguide body
    x_left = -0.5 * device.beam_length - extension_length

    # Rounded cap
    import numpy as np

    
    r = tip_width / 2
    n = 64
    theta = np.linspace(-np.pi/2, np.pi/2, n)
    cap = [(x_end + r*np.cos(t), r*np.sin(t)) for t in theta]

    # Full mask polygon
    taper_pts = [  
        (x_left,  -device.beam_width/2),
        (x_left,   device.beam_width/2),
        (x_start,  device.beam_width/2),
        (x_end,    tip_width/2),
    ] + cap + [
        (x_end,   -tip_width/2),
        (x_start, -device.beam_width/2),
    ]

    taper_mask = gdstk.Polygon(taper_pts)
    
    

    # -------------------------------------------------
    # 3) Apply AND (taper the waveguide body)
    # -------------------------------------------------
    core_wg_tapered = gdstk.boolean(
        core_wg,
        [taper_mask],
        "and",
        1e-9
    )

    # -------------------------------------------------
    # 4) Now union with coupler
    # -------------------------------------------------
    polys = gdstk.boolean(
        core_wg_tapered + [coupler_polys2],
        [],
        "or",
        1e-9,
        layer=0
    )
    
    for poly in polys:
        device_cell.add(poly)
    
    #merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
    
    
    #print("MERGED POLYS DUMP:")
    #for i, item in enumerate(merged):
        #print(i, type(item), item)

    
    #deviceunion = gdstk.boolean(beamcut, merged, "or", device.beam_layer)
    #for poly in merged:
        #device_cell.add(poly)
    '''        
    if(extensionparam == 0):
        #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
        #coupler_polys2 = createCoupler(filepath,-coupleroffset, 1, 4,0)
        wg_polys = [beam] + extensions
        merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
        for poly in merged:
            device_cell.add(poly)
        
    if(extensionparam == -1):
        #coupleroffset = 0.5*device.beam_length + 0.5*couplength
        #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
        #coupler_polys2 = createCoupler(filepath,-coupleroffset, 1, 4,0)
        wg_polys = [beam]
        merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
        for poly in merged:
            device_cell.add(poly)
    
    '''   
    # Holes
   
    #for h in holes:
        #device_cell.add(h)
    # Collect all polygons in this cell
    


    ref = gdstk.Reference(device_cell, (x, y))
    parent_cell.cell.add(ref)
    
    return parent_cell.cell


def placeBarebeam(parent_cell, device, mirrorholes, x, y, extensionparam, extension_length, taperdistance, coupler):
    # Unique cell name of the form "ARRAYA_DEVB_MC_X_Y"
    cell_name = f"{parent_cell.name}__{device.name}__M{mirrorholes}_{x}_{y}"
    device_cell = lib.new_cell(cell_name)

    # nanobeam
    beam = createNanobeam(
        length_um=device.beam_length,
        width_um=device.beam_width,
        layer=1
    )

    
    holes = createHoles(device, mirrorholes)
 
    

   

    
    # Waveguide extensions and couplers
    # Waveguide extensions and couplers
    extensions = createWaveguideextension(device, extension_length, taperdistance, extensionparam)
    filepath = coupler.filepath
    couplength = coupler.length
    coupleroffset = 0.5*device.beam_length + extension_length + 0.5*couplength
   
        
    #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
    #coupler_polys1.layer = 6
    
        
    coupler_polys2 = createCoupler(filepath, -coupleroffset, 1, 4, 0)
    coupler_polys2.layer = 7

    # -------------------------------------------------
    # 1) Merge beam + extensions first (no coupler)
    # -------------------------------------------------
    core_wg = gdstk.boolean(
        beam ,
        extensions,
        "or",
        1e-9
    )

    # -------------------------------------------------
    # 2) Build taper mask (modify as needed)
    # -------------------------------------------------
    taperdistance = 9
    taper_len   = taperdistance
    tip_width   = 0.15   # <-- your chosen final width (change this)

    # Rightmost edge of waveguide body (before coupler)
    x_end = 0.5 * device.beam_length + extension_length

    # Start of taper
    x_start = x_end - taper_len

    # Leftmost edge of waveguide body
    x_left = -0.5 * device.beam_length - extension_length

    # Rounded cap
    import numpy as np
    r = tip_width / 2
    n = 64
    theta = np.linspace(-np.pi/2, np.pi/2, n)
    cap = [(x_end + r*np.cos(t), r*np.sin(t)) for t in theta]

    # Full mask polygon
    taper_pts = [  
        (x_left,  -device.beam_width/2),
        (x_left,   device.beam_width/2),
        (x_start,  device.beam_width/2),
        (x_end,    tip_width/2),
    ] + cap + [
        (x_end,   -tip_width/2),
        (x_start, -device.beam_width/2),
    ]

    taper_mask = gdstk.Polygon(taper_pts)

    # -------------------------------------------------
    # 3) Apply AND (taper the waveguide body)
    # -------------------------------------------------
    core_wg_tapered = gdstk.boolean(
        core_wg,
        [taper_mask],
        "and",
        1e-9
    )

    # -------------------------------------------------
    # 4) Now union with coupler
    # -------------------------------------------------
    polys = gdstk.boolean(
        core_wg_tapered + [coupler_polys2],
        [],
        "or",
        1e-9,
        layer=0
    )
    
    for poly in polys:
        device_cell.add(poly)
        
        #merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
       
       
        #print("MERGED POLYS DUMP:")
        #for i, item in enumerate(merged):
            #print(i, type(item), item)

    
    #deviceunion = gdstk.boolean(beamcut, merged, "or", device.beam_layer)
    #for poly in merged:
        #device_cell.add(poly)
    '''        
    if(extensionparam == 0):
        #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
        #coupler_polys2 = createCoupler(filepath,-coupleroffset, 1, 4,0)
        wg_polys = [beam] + extensions
        merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
        for poly in merged:
            device_cell.add(poly)
        
    if(extensionparam == -1):
        #coupleroffset = 0.5*device.beam_length + 0.5*couplength
        #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
        #coupler_polys2 = createCoupler(filepath,-coupleroffset, 1, 4,0)
        wg_polys = [beam]
        merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
        for poly in merged:
            device_cell.add(poly)
    
    '''   
    # Holes
   
    #for h in holes:
        #device_cell.add(h)
    # Collect all polygons in this cell
    


    ref = gdstk.Reference(device_cell, (x, y))
    parent_cell.cell.add(ref)
    
    return parent_cell.cell




def placeBarebeamnocoup(parent_cell, device, mirrorholes, x, y, extensionparam, extension_length, taperdistance, coupler):
    # Unique cell name of the form "ARRAYA_DEVB_MC_X_Y"
    cell_name = f"{parent_cell.name}__{device.name}__M{mirrorholes}_{x}_{y}"
    device_cell = lib.new_cell(cell_name)

    # nanobeam
    beam = createNanobeam(
        length_um=device.beam_length,
        width_um=device.beam_width,
        layer=1
    )

    
    holes = createHoles(device, mirrorholes)
 
    

   

    
    # Waveguide extensions and couplers
    extensions = createWaveguideextension(device, extension_length, taperdistance, extensionparam)
    filepath = coupler.filepath
    #couplength = coupler.length
    #coupleroffset = 0.5*device.beam_length + extension_length + 0.5*couplength
    if(extensionparam == 1):
        
        core_wg = gdstk.boolean(
        beam ,
        extensions,
        "or",
        1e-9
    )

        # -------------------------------------------------
        # 2) Build taper mask (modify as needed)
        # -------------------------------------------------
        taperdistance = 4
        taper_len   = taperdistance
        tip_width   = 0.15   # <-- your chosen final width (change this)

        # Rightmost edge of waveguide body (before coupler)
        x_end = 0.5 * device.beam_length + extension_length

        # Start of taper
        x_start = x_end - taper_len

        # Leftmost edge of waveguide body
        x_left = -0.5 * device.beam_length - extension_length

        # Rounded cap
        import numpy as np
        # Create sharp taper
        sharp_pts = [
            (x_left,  device.beam_width/2),
            (x_start, device.beam_width/2),
            (x_end,   0), # Sharp tip at the center
            (x_start, -device.beam_width/2),
            (x_left,  -device.beam_width/2),
        ]
        taper_mask = gdstk.Polygon(sharp_pts)

        # Automatically round ONLY the tip (the 3rd point, index 2)
        # radius = tip_width / 2
        taper_mask.fillet([0, 0, tip_width/2, 0, 0], tolerance=1e-6)

        # -------------------------------------------------
        # 3) Apply AND (taper the waveguide body)
        # -------------------------------------------------
        polys = gdstk.boolean(
            core_wg,
            [taper_mask],
            "and",
            1e-9
        )

        # -------------------------------------------------
        # 4) Now union with coupler
        # -------------------------------------------------
        
        
        for poly in polys:
            device_cell.add(poly)
        
        #merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
       
       
        #print("MERGED POLYS DUMP:")
        #for i, item in enumerate(merged):
            #print(i, type(item), item)

    
    #deviceunion = gdstk.boolean(beamcut, merged, "or", device.beam_layer)
    #for poly in merged:
        #device_cell.add(poly)
    '''        
    if(extensionparam == 0):
        #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
        #coupler_polys2 = createCoupler(filepath,-coupleroffset, 1, 4,0)
        wg_polys = [beam] + extensions
        merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
        for poly in merged:
            device_cell.add(poly)
        
    if(extensionparam == -1):
        #coupleroffset = 0.5*device.beam_length + 0.5*couplength
        #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
        #coupler_polys2 = createCoupler(filepath,-coupleroffset, 1, 4,0)
        wg_polys = [beam]
        merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
        for poly in merged:
            device_cell.add(poly)
    
    '''   
    # Holes
   
    #for h in holes:
        #device_cell.add(h)
    # Collect all polygons in this cell
    


    ref = gdstk.Reference(device_cell, (x, y))
    parent_cell.cell.add(ref)
    
    return parent_cell.cell



def PlaceDeviceNoCoup(parent_cell, device, mirrorholes, x, y, extensionparam, extension_length, taperdistance, coupler):
    # Unique cell name of the form "ARRAYA_DEVB_MC_X_Y"
    cell_name = f"{parent_cell.name}__{device.name}__M{mirrorholes}_{x}_{y}"
    device_cell = lib.new_cell(cell_name)

    # nanobeam
    beam = createNanobeam(
        length_um=device.beam_length,
        width_um=device.beam_width,
        layer=1
    )

    
    holes = createHoles(device, mirrorholes)
 
    

    beamcut = gdstk.boolean([beam], holes, "not",1e-8, layer=3)

    
    # Waveguide extensions and couplers
    extensions = createWaveguideextension(device, extension_length, taperdistance, extensionparam)
    filepath = coupler.filepath
    #couplength = coupler.length
    #coupleroffset = 0.5*device.beam_length + extension_length + 0.5*couplength
    if(extensionparam == 1):
        core_wg = gdstk.boolean(
        beamcut + extensions,
        [],
        "or",
        1e-9
        )

        # -------------------------------------------------
        # 2) Build taper mask (modify as needed)
        # -------------------------------------------------
        taperdistance = 4
        taper_len   = taperdistance
        tip_width   = 0.15   # <-- your chosen final width (change this)

        # Rightmost edge of waveguide body (before coupler)
        x_end = 0.5 * device.beam_length + extension_length

        # Start of taper
        x_start = x_end - taper_len

        # Leftmost edge of waveguide body
        x_left = -0.5 * device.beam_length - extension_length

        # Rounded cap
        import numpy as np
        '''
        r = tip_width / 2
        n = 64
        theta = np.linspace(-np.pi/2, np.pi/2, n)
        cap = [(x_end + r*np.cos(t), r*np.sin(t)) for t in theta]

        # Full mask polygon
        taper_pts = [  
            (x_left,  -device.beam_width/2),
            (x_left,   device.beam_width/2),
            (x_start,  device.beam_width/2),
            (x_end,    tip_width/2),
        ] + cap + [
            (x_end,   -tip_width/2),
            (x_start, -device.beam_width/2),
        ]

        taper_mask = gdstk.Polygon(taper_pts)
        '''
        # Create sharp taper
        sharp_pts = [
            (x_left,  device.beam_width/2),
            (x_start, device.beam_width/2),
            (x_end,   0), # Sharp tip at the center
            (x_start, -device.beam_width/2),
            (x_left,  -device.beam_width/2),
        ]
        taper_mask = gdstk.Polygon(sharp_pts)

        # Automatically round ONLY the tip (the 3rd point, index 2)
        # radius = tip_width / 2
        taper_mask.fillet([0, 0, tip_width/2, 0, 0], tolerance=1e-6)

        # -------------------------------------------------
        # 3) Apply AND (taper the waveguide body)
        # -------------------------------------------------
        polys = gdstk.boolean(
            core_wg,
            [taper_mask],
            "and",
            1e-9
        )

        # -------------------------------------------------
        # 4) Now union with coupler
        # -------------------------------------------------
        
        
        for poly in polys:
            device_cell.add(poly)
        
        #merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
       
       
        #print("MERGED POLYS DUMP:")
        #for i, item in enumerate(merged):
            #print(i, type(item), item)

    
    #deviceunion = gdstk.boolean(beamcut, merged, "or", device.beam_layer)
    #for poly in merged:
        #device_cell.add(poly)
    '''        
    if(extensionparam == 0):
        #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
        #coupler_polys2 = createCoupler(filepath,-coupleroffset, 1, 4,0)
        wg_polys = [beam] + extensions
        merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
        for poly in merged:
            device_cell.add(poly)
        
    if(extensionparam == -1):
        #coupleroffset = 0.5*device.beam_length + 0.5*couplength
        #coupler_polys1 = createCoupler(filepath,coupleroffset, 0, 4,0)
        #coupler_polys2 = createCoupler(filepath,-coupleroffset, 1, 4,0)
        wg_polys = [beam]
        merged = gdstk.boolean(wg_polys, [], "or", layer = device.beam_layer)
        for poly in merged:
            device_cell.add(poly)
    
    '''   
    # Holes
   
    #for h in holes:
        #device_cell.add(h)
    # Collect all polygons in this cell
    


    ref = gdstk.Reference(device_cell, (x, y))
    parent_cell.cell.add(ref)
    
    return parent_cell.cell


In [9]:
def createUnitblock(array, devicee, scaling, x, y, extension_length):
    device = SimpleNamespace(**devicee.__dict__)   # ← local copy, one line

    device.ax *= scaling
    device.rx *= scaling
    device.ry *= scaling
    shiftscale = 2*device.beam_length + 117.4
    placeBarebeam(array, device, 0, x, y, 1, extension_length + 5.2 - grating.length, 2, grating)
    conn1 = createNanobeam(4, 0.5, 0)
    conn2 = createNanobeam(4, 0.5, 0)
    conn1.translate(x - (0.5*device.beam_length + extension_length + 5.2) - 2, y + 1.25)
    array.cell.add(conn1)
    conn2.translate(x + (0.5*device.beam_length + extension_length + 5.2) + 2, y + 1.25)
    array.cell.add(conn2)
    connn1 = createNanobeam(4, 0.5, 0)
    connn2 = createNanobeam(4, 0.5, 0)
    connn1.translate(x - (0.5*device.beam_length + extension_length + 5.2) - 2, y - 1.25)
    array.cell.add(connn1)
    connn2.translate(x + (0.5*device.beam_length + extension_length + 5.2) + 2, y - 1.25)
    array.cell.add(connn2)
    beam1 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam2 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam1.translate(x - (0.5*device.beam_length + extension_length + 5.2 + 10) - 4, y - 75)
    beam2.translate(x + (0.5*device.beam_length + extension_length + 5.2 + 10) + 4, y - 75)
    merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    shape2 = merged2[0]
    shape2.fillet(0.8)
    array.cell.add(shape2)
    mergedd1 = gdstk.boolean([beam1], [connn1], "or", layer=0)
    shape1 = mergedd1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged2 = gdstk.boolean([beam2], [connn2], "or", layer=0)
    shape2 = merged2[0]
    shape2.fillet(0.8)
    array.cell.add(shape2)
    #array.cell.add(beam1)
    #array.cell.add(beam2)
    i=1
    for j in range(0, 10, 2):
        placeDevice(array, device, j, x, y - 30*(i), 1, extension_length + 5.2 - grating.length, 2, grating)
        conn1 = createNanobeam(4, 0.5, 0)
        conn2 = createNanobeam(4, 0.5, 0)
        conn1.translate(x - (0.5*device.beam_length + extension_length + 5.2) - 2, y - 30*i + 1.25)
        array.cell.add(conn1)
        conn2.translate(x + (0.5*device.beam_length + extension_length + 5.2) + 2, y - 30*i - 1.25)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape2 = merged2[0]
        shape2.fillet(0.8)
        array.cell.add(shape2)
        conn1 = createNanobeam(4, 0.5, 0)
        conn2 = createNanobeam(4, 0.5, 0)
        conn1.translate(x - (0.5*device.beam_length + extension_length + 5.2) - 2, y - 30*i - 1.25)
        array.cell.add(conn1)
        conn2.translate(x + (0.5*device.beam_length + extension_length + 5.2) + 2, y - 30*i + 1.25)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape2 = merged2[0]
        shape2.fillet(0.8)
        array.cell.add(shape2)
        i=i+1

    
    beamm1 = createNanobeam(device.beam_length+ 2*extension_length + 2*grating.length +4, 8.9*2 + 5*30 + 6*device.beam_width, 8)
    beamm1.translate(x, y - 75)
    array.cell.add(beamm1)
    beammm1 = createNanobeam(device.beam_length+ extension_length-2, 30-5.7, 4)
    beammm1.translate(x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ extension_length-2, 30-5.7, 4)
    beammm2.translate(x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ extension_length-2, 30-5.7, 4)
    beammm3.translate(x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ extension_length-2, 30-5.7, 4)
    beammm4.translate(x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ extension_length-2, 30-5.7, 4)
    beammm5.translate(x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ extension_length-2, 30-5.7, 4)
    beammm6.translate(x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length + extension_length-2, 30-5.7, 4)
    beammm7.translate(x, y-165)
    array.cell.add(beammm7)
  

  
    beammm1 = createNanobeam(device.beam_length+ 2*extension_length + 2*grating.length +4, 30-grating.length - 4, 4)
    beammm1.translate(x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ 2*extension_length + 2*grating.length +4, 30-grating.length - 4, 4)
    beammm2.translate(x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ 2*extension_length + 2*grating.length +4, 30-grating.length - 4, 4)
    beammm3.translate(x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ 2*extension_length + 2*grating.length +4, 30-grating.length - 4, 4)
    beammm4.translate(x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ 2*extension_length + 2*grating.length +4, 30-grating.length - 4, 4)
    beammm5.translate(x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ 2*extension_length + 2*grating.length +4, 30-grating.length - 4, 4)
    beammm6.translate(x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length+ 2*extension_length + 2*grating.length +4, 30-grating.length - 4, 4)
    beammm7.translate(x, y-165)
    array.cell.add(beammm7)

    

    extension_lengthh = 0.5*(2*extension_length + grating.length - gratingminus10.length)
   
    placeBarebeam(array, device, 0, x + shiftscale, y, 1, extension_lengthh, 1, gratingminus10)
    conn1 = createNanobeam(4, 0.5, 0)
    conn2 = createNanobeam(4, 0.5, 0)
    conn1.translate(shiftscale + x - (0.5*device.beam_length + extension_lengthh + gratingminus10.length) - 2, y+1.25)
    array.cell.add(conn1)
    conn2.translate(shiftscale + x + (0.5*device.beam_length + extension_lengthh + gratingminus10.length) + 2, y-1.25)
    array.cell.add(conn2)
    beam1 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam2 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam1.translate(shiftscale+x - (0.5*device.beam_length + extension_lengthh + gratingminus10.length + 10) - 4, y - 75)
    beam2.translate(shiftscale+x + (0.5*device.beam_length + extension_lengthh + gratingminus10.length  + 10) + 4, y - 75)
    #array.cell.add(beam1)
    #array.cell.add(beam2)
    merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
    shape1 = merged1[0] 
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    shape1 = merged2[0] 
    shape1.fillet(0.8)
    array.cell.add(shape1)
    conn1 = createNanobeam(4, 0.5, 0)
    conn2 = createNanobeam(4, 0.5, 0)
    conn1.translate(shiftscale + x - (0.5*device.beam_length + extension_lengthh + gratingminus10.length) - 2, y-1.25)
    array.cell.add(conn1)
    conn2.translate(shiftscale + x + (0.5*device.beam_length + extension_lengthh + gratingminus10.length) + 2, y+1.25)
    array.cell.add(conn2)
    beam1 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam2 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam1.translate(shiftscale+x - (0.5*device.beam_length + extension_lengthh + gratingminus10.length + 10) - 4, y - 75)
    beam2.translate(shiftscale+x + (0.5*device.beam_length + extension_lengthh + gratingminus10.length  + 10) + 4, y - 75)
    #array.cell.add(beam1)
    #array.cell.add(beam2)
    merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
    shape1 = merged1[0] 
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    shape1 = merged2[0] 
    shape1.fillet(0.8)
    array.cell.add(shape1)
    i=1
    for j in range(0, 10, 2):
        placeDevice(array, device, j, x + shiftscale, y - 30*(i), 1, extension_lengthh, 1, gratingminus10)
        conn1 = createNanobeam(4, 0.5, 0)
        conn2 = createNanobeam(4, 0.5, 0)
        conn1.translate(x +shiftscale - (0.5*device.beam_length + extension_lengthh + gratingminus10.length) - 2, y - 30*i + 1.25)
        array.cell.add(conn1)
        conn2.translate(x + shiftscale + (0.5*device.beam_length + extension_lengthh + gratingminus10.length) + 2, y - 30*i - 1.25)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0] 
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape1 = merged2[0] 
        shape1.fillet(0.8)
        array.cell.add(shape1)
        conn1 = createNanobeam(4, 0.5, 0)
        conn2 = createNanobeam(4, 0.5, 0)
        conn1.translate(x +shiftscale - (0.5*device.beam_length + extension_lengthh + gratingminus10.length) - 2, y - 30*i - 1.25)
        array.cell.add(conn1)
        conn2.translate(x + shiftscale + (0.5*device.beam_length + extension_lengthh + gratingminus10.length) + 2, y - 30*i + 1.25)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0] 
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape1 = merged2[0] 
        shape1.fillet(0.8)
        array.cell.add(shape1)
        i=i+1
    
    beamm1 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingminus10.length +4, 8.9*2 + 5*30 + 6*device.beam_width, 8)
    beamm1.translate(shiftscale+x, y - 75)
    array.cell.add(beamm1)
    beammm1 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm1.translate(shiftscale+x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm2.translate(shiftscale+x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm3.translate(shiftscale+x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm4.translate(shiftscale+x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm5.translate(shiftscale+x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm6.translate(shiftscale+x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length + extension_lengthh-2, 30-5.7, 4)
    beammm7.translate(shiftscale+x, y-165)
    array.cell.add(beammm7)
  
    
  
    beammm1 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingminus10.length +4, 30-gratingminus10.length - 4, 4)
    beammm1.translate(shiftscale+x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingminus10.length +4, 30-gratingminus10.length - 4, 4)
    beammm2.translate(shiftscale+x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingminus10.length +4, 30-gratingminus10.length - 4, 4)
    beammm3.translate(shiftscale+x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingminus10.length +4, 30-gratingminus10.length - 4, 4)
    beammm4.translate(shiftscale+x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingminus10.length +4, 30-gratingminus10.length - 4, 4)
    beammm5.translate(shiftscale+x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingminus10.length +4, 30-gratingminus10.length - 4, 4)
    beammm6.translate(shiftscale+x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingminus10.length +4, 30-gratingminus10.length - 4, 4)
    beammm7.translate(shiftscale+x, y-165)
    array.cell.add(beammm7)


    extension_lengthh = 0.5*(2*extension_length + grating.length - gratingplus10.length)
    placeBarebeam(array, device, 0, x + 2*shiftscale, y, 1, extension_lengthh, 1, gratingplus10)
    conn1 = createNanobeam(4, 0.5, 0)
    conn2 = createNanobeam(4, 0.5, 0)
    conn1.translate(2*shiftscale + x - (0.5*device.beam_length + extension_lengthh + gratingplus10.length) - 2, y + 1.25)
    array.cell.add(conn1)
    conn2.translate(2*shiftscale + x + (0.5*device.beam_length + extension_lengthh + gratingplus10.length) + 2, y -1.25)
    array.cell.add(conn2)
    beam1 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam2 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam1.translate(2*shiftscale+x - (0.5*device.beam_length + extension_lengthh + gratingplus10.length + 10) - 4, y - 75)
    beam2.translate(2*shiftscale+x + (0.5*device.beam_length + extension_lengthh + gratingplus10.length + 10) + 4, y - 75)
    #array.cell.add(beam1)
    #array.cell.add(beam2)
    merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged1 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    conn1 = createNanobeam(4, 0.5, 0)
    conn2 = createNanobeam(4, 0.5, 0)
    conn1.translate(2*shiftscale + x - (0.5*device.beam_length + extension_lengthh + gratingplus10.length) - 2, y - 1.25)
    array.cell.add(conn1)
    conn2.translate(2*shiftscale + x + (0.5*device.beam_length + extension_lengthh + gratingplus10.length) + 2, y +1.25)
    array.cell.add(conn2)
    beam1 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam2 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam1.translate(2*shiftscale+x - (0.5*device.beam_length + extension_lengthh + gratingplus10.length + 10) - 4, y - 75)
    beam2.translate(2*shiftscale+x + (0.5*device.beam_length + extension_lengthh + gratingplus10.length + 10) + 4, y - 75)
    #array.cell.add(beam1)
    #array.cell.add(beam2)
    merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged1 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    i=1
    for j in range(0, 10, 2):
        placeDevice(array, device, j, x + 2*shiftscale, y - 30*(i), 1, extension_lengthh, 1, gratingplus10)
        conn1 = createNanobeam(4, 0.5, 0)
        conn2 = createNanobeam(4, 0.5, 0)
        conn1.translate(x +2*shiftscale - (0.5*device.beam_length + extension_lengthh + gratingplus10.length) - 2, y - 30*i + 1.25)
        array.cell.add(conn1)
        conn2.translate(x + 2*shiftscale + (0.5*device.beam_length + extension_lengthh + gratingplus10.length) + 2, y - 30*i - 1.25)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged1 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)
        conn1 = createNanobeam(4, 0.5, 0)
        conn2 = createNanobeam(4, 0.5, 0)
        conn1.translate(x +2*shiftscale - (0.5*device.beam_length + extension_lengthh + gratingplus10.length) - 2, y - 30*i - 1.25)
        array.cell.add(conn1)
        conn2.translate(x + 2*shiftscale + (0.5*device.beam_length + extension_lengthh + gratingplus10.length) + 2, y - 30*i + 1.25)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged1 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)

        i=i+1
    
    beamm1 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingplus10.length +4, 8.9*2 + 5*30 + 6*device.beam_width, 8)
    beamm1.translate(2*shiftscale+x, y - 75)
    array.cell.add(beamm1)
    beammm1 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm1.translate(2*shiftscale+x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm2.translate(2*shiftscale+x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm3.translate(2*shiftscale+x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm4.translate(2*shiftscale+x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm5.translate(2*shiftscale+x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ extension_lengthh-2, 30-5.7, 4)
    beammm6.translate(2*shiftscale+x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length + extension_lengthh-2, 30-5.7, 4)
    beammm7.translate(2*shiftscale+x, y-165)
    array.cell.add(beammm7)
  

  
    beammm1 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingplus10.length +4, 30-gratingplus10.length - 4, 4)
    beammm1.translate(2*shiftscale+x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingplus10.length +4, 30-gratingplus10.length - 4, 4)
    beammm2.translate(2*shiftscale+x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingplus10.length +4, 30-gratingplus10.length - 4, 4)
    beammm3.translate(2*shiftscale+x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingplus10.length +4, 30-gratingplus10.length - 4, 4)
    beammm4.translate(2*shiftscale+x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingplus10.length +4, 30-gratingplus10.length - 4, 4)
    beammm5.translate(2*shiftscale+x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingplus10.length +4, 30-gratingplus10.length - 4, 4)
    beammm6.translate(2*shiftscale+x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length+ 2*extension_lengthh + 2*gratingplus10.length +4, 30-gratingplus10.length - 4, 4)
    beammm7.translate(2*shiftscale+x, y-165)
    array.cell.add(beammm7)


    
    label = f"{device.name} S={scaling}"
    text_poly = gdstk.text(label,20,(x-53.7, y + 100),layer=0,datatype=0)
    array.cell.add(*text_poly)
     
    return gdstk.Reference(array.cell, (0, 0))





















    
def createUnitBlockNoCoupler(array, devicee, scaling, x, y, extension_length):
    device = SimpleNamespace(**devicee.__dict__)   # ← local copy, one line

    device.ax *= scaling
    device.rx *= scaling
    device.ry *= scaling
    shiftscale = 2*device.beam_length + 117.4
    beam1 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam2 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam1.translate(x - (0.5*device.beam_length + extension_length + 10) - 4, y - 75)
    beam2.translate(x + (0.5*device.beam_length + extension_length + 10) + 4, y - 75)
   
    array.cell.add(beam1)
    array.cell.add(beam2)

    placeBarebeamnocoup(array, device, 0, x, y - 30*(0), 1, extension_length, 2, grating)
    conn1 = createNanobeam(4, device.beam_width, 0)
    conn2 = createNanobeam(4, device.beam_width, 0)
    conn1.translate(x - (0.5*device.beam_length + extension_length) - 2, y - 30*0)
    array.cell.add(conn1)
    conn2.translate(x + (0.5*device.beam_length + extension_length) + 2, y - 30*0)
    array.cell.add(conn2)
    merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    shape2 = merged2[0]
    shape2.fillet(0.8)
    array.cell.add(shape2)
    i=1
    for j in range(0, 10, 2):
        PlaceDeviceNoCoup(array, device, j, x, y - 30*(i), 1, extension_length, 2, grating)
        conn1 = createNanobeam(4, device.beam_width, 0)
        conn2 = createNanobeam(4, device.beam_width, 0)
        conn1.translate(x - (0.5*device.beam_length + extension_length) - 2, y - 30*i)
        array.cell.add(conn1)
        conn2.translate(x + (0.5*device.beam_length + extension_length) + 2, y - 30*i)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape2 = merged2[0]
        shape2.fillet(0.8)
        array.cell.add(shape2)
        i=i+1


    beamm1 = createNanobeam(device.beam_length+ 2*extension_length +4, 8.9*2 + 5*30 + 6*device.beam_width, 8)
    beamm1.translate(x, y - 75)
    array.cell.add(beamm1)
    beammm1 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm1.translate(x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm2.translate(x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm3.translate(x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm4.translate(x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm5.translate(x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm6.translate(x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length + extension_length+2, 30-5.7, 4)
    beammm7.translate(x, y-165)
    array.cell.add(beammm7)
    x = x+shiftscale
    beam1 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam2 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam1.translate(x - (0.5*device.beam_length + extension_length + 10) - 4, y - 75)
    beam2.translate(x + (0.5*device.beam_length + extension_length + 10) + 4, y - 75)
   
    array.cell.add(beam1)
    array.cell.add(beam2)

    placeBarebeamnocoup(array, device, 0, x, y - 30*(0), 1, extension_length, 2, grating)
    conn1 = createNanobeam(4, device.beam_width, 0)
    conn2 = createNanobeam(4, device.beam_width, 0)
    conn1.translate(x - (0.5*device.beam_length + extension_length) - 2, y - 30*0)
    array.cell.add(conn1)
    conn2.translate(x + (0.5*device.beam_length + extension_length) + 2, y - 30*0)
    array.cell.add(conn2)
    merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    shape2 = merged2[0]
    shape2.fillet(0.8)
    array.cell.add(shape2)
    i=1
    for j in range(0, 10, 2):
        PlaceDeviceNoCoup(array, device, j, x, y - 30*(i), 1, extension_length, 2, grating)
        conn1 = createNanobeam(4, device.beam_width, 0)
        conn2 = createNanobeam(4, device.beam_width, 0)
        conn1.translate(x - (0.5*device.beam_length + extension_length) - 2, y - 30*i)
        array.cell.add(conn1)
        conn2.translate(x + (0.5*device.beam_length + extension_length) + 2, y - 30*i)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape2 = merged2[0]
        shape2.fillet(0.8)
        array.cell.add(shape2)
        i=i+1


    beamm1 = createNanobeam(device.beam_length+ 2*extension_length +4, 8.9*2 + 5*30 + 6*device.beam_width, 8)
    beamm1.translate(x, y - 75)
    array.cell.add(beamm1)
    beammm1 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm1.translate(x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm2.translate(x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm3.translate(x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm4.translate(x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm5.translate(x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm6.translate(x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length + extension_length+2, 30-5.7, 4)
    beammm7.translate(x, y-165)
    array.cell.add(beammm7)

    x = x+shiftscale
    beam1 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam2 = createNanobeam(20, 8.9*2 + 5*30 + 6*device.beam_width, 0)
    beam1.translate(x - (0.5*device.beam_length + extension_length + 10) - 4, y - 75)
    beam2.translate(x + (0.5*device.beam_length + extension_length + 10) + 4, y - 75)
   
    array.cell.add(beam1)
    array.cell.add(beam2)


    placeBarebeamnocoup(array, device, 0, x, y - 30*(0), 1, extension_length, 2, grating)
    conn1 = createNanobeam(4, device.beam_width, 0)
    conn2 = createNanobeam(4, device.beam_width, 0)
    conn1.translate(x - (0.5*device.beam_length + extension_length) - 2, y - 30*0)
    array.cell.add(conn1)
    conn2.translate(x + (0.5*device.beam_length + extension_length) + 2, y - 30*0)
    array.cell.add(conn2)
    merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    shape2 = merged2[0]
    shape2.fillet(0.8)
    array.cell.add(shape2)
    i=1
    for j in range(0, 10, 2):
        PlaceDeviceNoCoup(array, device, j, x, y - 30*(i), 1, extension_length, 2, grating)
        conn1 = createNanobeam(4, device.beam_width, 0)
        conn2 = createNanobeam(4, device.beam_width, 0)
        conn1.translate(x - (0.5*device.beam_length + extension_length) - 2, y - 30*i)
        array.cell.add(conn1)
        conn2.translate(x + (0.5*device.beam_length + extension_length) + 2, y - 30*i)
        array.cell.add(conn2)
        merged1 = gdstk.boolean([beam1], [conn1], "or", layer=0)
        shape1 = merged1[0]
        shape1.fillet(0.8)
        array.cell.add(shape1)
        merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
        shape2 = merged2[0]
        shape2.fillet(0.8)
        array.cell.add(shape2)
        i=i+1


    beamm1 = createNanobeam(device.beam_length+ 2*extension_length +4, 8.9*2 + 5*30 + 6*device.beam_width, 8)
    beamm1.translate(x, y - 75)
    array.cell.add(beamm1)
    beammm1 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm1.translate(x, y+15)
    array.cell.add(beammm1)
    beammm2 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm2.translate(x, y-15)
    array.cell.add(beammm2)
    beammm3 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm3.translate(x, y-45)
    array.cell.add(beammm3)
    beammm4 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm4.translate(x, y-75)
    array.cell.add(beammm4)
    beammm5 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm5.translate(x, y-105)
    array.cell.add(beammm5)
    beammm6 = createNanobeam(device.beam_length+ extension_length+2, 30-5.7, 4)
    beammm6.translate(x, y-135)
    array.cell.add(beammm6)
    beammm7 = createNanobeam(device.beam_length + extension_length+2, 30-5.7, 4)
    beammm7.translate(x, y-165)
    array.cell.add(beammm7)


    label = f"{device.name} S={scaling} NoCoupler"
    text_poly = gdstk.text(label,20,(x-53.7 - 2*shiftscale, y + 100),layer=0,datatype=0)
    array.cell.add(*text_poly)

    return gdstk.Reference(array.cell, (0, 0))



In [10]:
def createDeviceBare(array, devicee, scaling, x, y, extension_length):
    device = SimpleNamespace(**devicee.__dict__)   # ← local copy, one line
    # The "Pythonic" way to rotate (x, y) -> (-y, x)
    x = x-4.6 -10
    extension_length = extension_length 
    device.ax *= scaling
    device.rx *= scaling
    device.ry *= scaling
    shiftscale = 2*device.beam_length + 117.4
    barebeam = placeBarebeamnocoup(array, device, 0, x, y, 1, extension_length + 5.2 - grating.length, 2, grating)
    #conn1 = createNanobeam(4, 0.5, 0)
    #conn2 = createNanobeam(4, 0.5, 0)
    #conn1.translate( x - (0.5*device.beam_length + extension_length + 5.2) - 2, y + 1.25)
    #array.cell.add(conn1)
    #conn2.translate(x + (0.5*device.beam_length + extension_length + 5.2) + 2, y + 1.25)
    #array.cell.add(conn2)
    #connn1 = createNanobeam(4, 0.5, 0)
    #connn2 = createNanobeam(4, 0.5, 0)
    #connn1.translate( x - (0.5*device.beam_length + extension_length + 5.2) - 2, y - 1.25)
    #array.cell.add(connn1)
    #connn2.translate(x + (0.5*device.beam_length + extension_length + 5.2) + 2, y - 1.25)
    #array.cell.add(connn2)
    beam1 = createNanobeam(40, 97.103998, 0)
    #beam2 = createNanobeam(40, 40, 0)
    beam1.translate(x + 4.6  - (0.5*device.beam_length + extension_length - 4.6 + 5.2 + 20) - 4, y )
    beam2 = createNanobeam(2, device.beam_width, 0)
    beam2.translate(x + 4.6  - (0.5*device.beam_length + extension_length - 4.6 + 5.2 + 20) - 4 + 20 + 1, y )
    #array.cell.add(beam1)
    merged1 = gdstk.boolean([beam1], [beam2], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    array.cell.add(beam2)
    array.cell.add(beam1)
    #merged1 = gdstk.boolean([beam1], [barebeam], "or", layer=0)
    #shape1 = merged1[0]
    #shape1.fillet(0.8)
    #array.cell.add(shape1)



    #merged2 = gdstk.boolean([beam2], [conn2], "or", layer=0)
    #shape2 = merged2[0]
    #shape2.fillet(0.8)
    #array.cell.add(shape2)
    #mergedd1 = gdstk.boolean([beam1], [connn1], "or", layer=0)
    #shape1 = mergedd1[0]
    #shape1.fillet(0.8)
    #array.cell.add(shape1)
    #merged2 = gdstk.boolean([beam2], [connn2], "or", layer=0)
    #shape2 = merged2[0]
    #shape2.fillet(0.8)
    #array.cell.add(shape2)
    #array.cell.add(beam1)
    #array.cell.add(beam2)
    

    
    


    
    label = f"{device.name} S={scaling}"
    text_poly = gdstk.text(label,20,(x-350, y-10),layer=4,datatype=0)
    array.cell.add(*text_poly)
     
    return gdstk.Reference(array.cell, (0, 0))
def createDevice(array, devicee, scaling, x, y, extension_length):
    x = x-4.6 - 10
    extension_length = extension_length
    device = SimpleNamespace(**devicee.__dict__)   # ← local copy, one line
    
    device.ax *= scaling
    #device.rx *= scaling
    #device.ry *= scaling
    shiftscale = 2*device.beam_length + 117.4
    PlaceDeviceNoCoup(array, device, 5, x, y, 1, extension_length + 5.2 - grating.length, 2, grating)
    #conn1 = createNanobeam(4, 0.5, 0)
    #conn2 = createNanobeam(4, 0.5, 0)
    #conn1.translate( x - (0.5*device.beam_length + extension_length + 5.2) - 2, y + 1.25)
    #array.cell.add(conn1)
    #conn2.translate(x + (0.5*device.beam_length + extension_length + 5.2) + 2, y + 1.25)
    #array.cell.add(conn2)
    #connn1 = createNanobeam(4, 0.5, 0)
    #connn2 = createNanobeam(4, 0.5, 0)
    #connn1.translate( x - (0.5*device.beam_length + extension_length + 5.2) - 2, y - 1.25)
    #array.cell.add(connn1)
    #connn2.translate(x + (0.5*device.beam_length + extension_length + 5.2) + 2, y - 1.25)
    #array.cell.add(connn2)
    beam1 = createNanobeam(40, 97.103998, 0)
    #beam2 = createNanobeam(40, 40, 0)
    beam1.translate(x + 4.6 - (0.5*device.beam_length + extension_length - 4.6 + 5.2 + 20) - 4, y )
    
    beam2 = createNanobeam(2, device.beam_width, 0)
    beam2.translate(x + 4.6 - (0.5*device.beam_length + extension_length - 4.6 + 5.2 + 20) - 4 + 20 + 1, y )
    #array.cell.add(beam1)
    merged1 = gdstk.boolean([beam1], [beam2], "or", layer=0)
    shape1 = merged1[0]
    shape1.fillet(0.8)
    array.cell.add(shape1)
    array.cell.add(beam2)
    array.cell.add(beam1)

    
    


    
    label = f"{device.name} S={scaling}"
    text_poly = gdstk.text(label,20,(x-350, y-10),layer=4,datatype=0)
    array.cell.add(*text_poly)
     
    return gdstk.Reference(array.cell, (0, 0))

In [11]:
def clear_layer_2_globally(x_center, y_center, width, height):
    target_cell = parentglobal.cell 
    
    eraser = gdstk.rectangle((-width/2, -height/2), (width/2, height/2), layer=2)
    eraser.translate(x_center, y_center)

    # 2. Now target_cell.polygons is NOT empty because we injected them!
    layer2_polys = [p for p in target_cell.polygons if p.layer == 2]

    if layer2_polys:
        # 3. Perform the subtraction
        new_polys = gdstk.boolean(layer2_polys, [eraser], "not", layer=2)
        
        # 4. Wipe the old Layer 2 and add the punched results
        target_cell.filter([(2, 0)], "not")
        if new_polys:
            target_cell.add(*new_polys)
def etchadd(x,y,h,w):
    rect = gdstk.rectangle((-w/2, -h/2), (w/2, h/2), layer=2)
    rect.translate(x, y)
    parentglobal.cell.add(rect)

In [12]:
def etchadjust(x,y):
    #clear_layer_2_globally(x-30.5, y+8.05, 5.2, 1.105)
    
    x = x-4.6 + 2 - 10
    elength = 3.9 + 4.6
    etchadd(x + 4.6 - 2 - (0.5*28 + elength - 4.6 + 5.2 + 20) - 4 + 20 + (45/2) + 6, y + 0.85 + 7, 2, 45)
    etchadd(x + 4.6 - 2 - (0.5*28 + elength - 4.6 + 5.2 + 20) - 4+20+(45/2) + 6, y - 0.85 - 7, 2,45)
   
    
    


# Generate Single Sided GDS

In [13]:

#old_top_cell = lib.cells[0]  
#old_top_cell.add(gdstk.Reference(parentglobal.cell, (0, 0)))
output_path = r"C:\Users\Connor\Documents\GDSPhotonics\micron45FabRun2SingleSidedTest12.gds"
#cell1 = createDeviceBare(A1, device3, 1.0, 2837.1, 5701.3, 3.9)
#etchadjust(2837.1, 5701.3)
for i in range(52):
    q = 2837.1
    q = q-4.6 + 2 - 10
    elength = 3.9 + 4.6
    etchadd(q + 4.6 - 2 - (0.5*28 + elength - 4.6 + 5.2 + 20) - 4 + 20 + (65/2) + 6, 5701.3 - i*96.15, 15, 65)
    clear_layer_2_globally(q + 4.6 - 2 - (0.5*28+ elength - 4.6 + 5.2 + 20) - 4 + 20 + (45/2) + 6 - 0.8,5701.3 - i*96.15, 45, 15)
    



# Build ordered list of namespace objects (NOT strings)
names = [
    A1,
    A11,
    A111,
    A1111,
    A11111,
    A111111,
    A1111111,
    A11111111,
    A111111111,
    A1111111111,
    B1,
    B11,
    B111,
]


x0 = 2837.1
y0 = 5701.3
dy = 96.15


scale0 = 1.0


n_sweep = 12
scales_sweep = [
    round(0.95 + k*(0.10/(n_sweep - 1)), 3)
    for k in range(n_sweep)
]


for i in range(13):
    name = names[i]
    x = x0
    y = y0 - i * dy 

    
    

    if i == 0:
        
        cell1 = createDeviceBare(A1, device1, scale0, x, y, 0)
        etchadjust(x,y)
    else:
        
        scale = scales_sweep[i - 1]  
        cell1 = createDevice(name, device1, scale, x, y, 0)
        etchadjust(x,y)
        #clear_layer_2_globally(x-30.5, y+8.03 - i*0.05, 5.2, 1.12)
        #clear_layer_2_globally(x-30.5, y-8.03 + i*0.05, 5.2, 1.12)
        #etchadd( x+3, y + 0.85 + 7 - i*0.05, 2, 48 + 2*3.9 + 6)
        #etchadd( x+3, y - 0.85 - 7 + i*0.05, 2,48 + 2*3.9 + 6)
# Build ordered list of namespace objects (NOT strings)
names = [
    B1111,
    B11111,
    B11111,
    B1111111,
    B11111111,
    B111111111,
    B1111111111,
    C1,
    C11,
    C111,
    R1,
    R11,
    R111,
]


x0 = 2837.1
y0 = 4451.35
dy = 96.15


scale0 = 1.0


n_sweep = 12
scales_sweep = [
    round(0.95 + k*(0.10/(n_sweep - 1)), 3)
    for k in range(n_sweep)
]


for i in range(13):
    name = names[i]
    x = x0
    y = y0 - i * dy 

    
    

    if i == 0:
        
        cell1 = createDeviceBare(F1, device2, scale0, x, y, 0)
        etchadjust(x,y)
    else:
        
        scale = scales_sweep[i - 1]  
        cell1 = createDevice(name, device2, scale, x, y, 0)
        etchadjust(x,y)
        #clear_layer_2_globally(x-30.5, y+8.03 - i*0.05, 5.2, 1.12)
        #clear_layer_2_globally(x-30.5, y-8.03 + i*0.05, 5.2, 1.12)
        #etchadd( x+3, y + 0.85 + 7 - i*0.05, 2, 48 + 2*3.9 + 6)
        #etchadd( x+3, y - 0.85 - 7 + i*0.05, 2,48 + 2*3.9 + 6)
# Build ordered list of namespace objects (NOT strings)
names = [
    C1111,
    C11111,
    C11111,
    C1111111,
    C11111111,
    C111111111,
    C1111111111,
    D1,
    D11,
    D111,
    Q1,
    Q11,
    Q111,
]


x0 = 2837.1
y0 = 3201.4
dy = 96.15


scale0 = 1.0


n_sweep = 12
scales_sweep = [
    round(0.95 + k*(0.10/(n_sweep - 1)), 3)
    for k in range(n_sweep)
]


for i in range(13):
    name = names[i]
    x = x0
    y = y0 - i * dy 

    
    

    if i == 0:
        
        cell1 = createDeviceBare(F11, device3, scale0, x, y, 0)
        etchadjust(x,y)
    else:
        
        scale = scales_sweep[i - 1]  
        cell1 = createDevice(name, device3, scale, x, y, 0)
        etchadjust(x,y)
        #clear_layer_2_globally(x-30.5, y+8.03 - i*0.05, 5.2, 1.12)
        #clear_layer_2_globally(x-30.5, y-8.03 + i*0.05, 5.2, 1.12)
        #etchadd( x+3, y + 0.85 + 7 - i*0.05, 2, 48 + 2*3.9 + 6)
        #etchadd( x+3, y - 0.85 - 7 + i*0.05, 2,48 + 2*3.9 + 6)
names = [
    D1111,
    D11111,
    D11111,
    D1111111,
    D11111111,
    D111111111,
    D1111111111,
    E1,
    E11,
    E111,
    J1,
    J11,
    J111,
]


x0 = 2837.1
y0 = 1951.45
dy = 96.15


scale0 = 1.0


n_sweep = 12
scales_sweep = [
    round(0.95 + k*(0.10/(n_sweep - 1)), 3)
    for k in range(n_sweep)
]


for i in range(13):
    name = names[i]
    x = x0
    y = y0 - i * dy 

    
    

    if i == 0:
        
        cell1 = createDeviceBare(F111, device4, scale0, x, y, 0)
        etchadjust(x,y)
    else:
        
        scale = scales_sweep[i - 1]  
        cell1 = createDevice(name, device4, scale, x, y, 0)
        etchadjust(x,y)
        #clear_layer_2_globally(x-30.5, y+8.03 - i*0.05, 5.2, 1.12)
        #clear_layer_2_globally(x-30.5, y-8.03 + i*0.05, 5.2, 1.12)
        #etchadd( x+3, y + 0.85 + 7 - i*0.05, 2, 48 + 2*3.9 + 6)
        #etchadd( x+3, y - 0.85 - 7 + i*0.05, 2,48 + 2*3.9 + 6)




label = f"2-26 Single-Side Photonic Crystals"
xy_center = (1500, 6250)

# 1. Create the text polygons
text_poly_list = gdstk.text(label, 150, (0, 0), layer=4, datatype=0)

# 2. Calculate the bounding box of the created text polygons
# Combine all polygons into a single list for bounding box calculation
all_points = []
for poly in text_poly_list:
    all_points.extend(poly.points)

# Find min/max x and y
min_x = min(p[0] for p in all_points)
max_x = max(p[0] for p in all_points)
min_y = min(p[1] for p in all_points)
max_y = max(p[1] for p in all_points)

# 3. Calculate the current center of the text
current_center_x = (min_x + max_x) / 2
current_center_y = (min_y + max_y) / 2

# 4. Calculate the required translation
translation_x = xy_center[0] - current_center_x
translation_y = xy_center[1] - current_center_y

# 5. Translate each polygon to the desired center point
for poly in text_poly_list:
    poly.translate(translation_x, translation_y)

# 6. Add the polygons to the cell
parentglobal.cell.add(*text_poly_list)


lib.write_gds(output_path)
print(f"Wrote {output_path}")

Wrote C:\Users\Connor\Documents\GDSPhotonics\micron45FabRun2SingleSidedTest12.gds


C:\Users\Connor\AppData\Local\Temp\ipykernel_16356\2931860232.py:269: RuntimeWarning: Saved file uses unofficially supported extensions.
  lib.write_gds(output_path)


# Generate GDS


# Import Libraries
